In [1]:
import pandas as pd
import numpy as np
import json
import ast  # to safely convert string to dictionary
import datetime
import geopandas as gpd
from shapely.geometry import Point
from pdfs_to_text import pdfs_downloader, pdfs_preprocessing

# Downloading PDFs

In the first notebooks, we obtained the information about building plans and the links to the respective PDFs. In this notebook, you will see how to use the function that takes as input that metadata and downloads all PDFs.

First, read the metadata:

In [2]:
filename = '../data/raw/geoservices_results/map-data.jsonl'

In [3]:
data = []
buffer = ""
with open(filename, 'r', encoding='utf-8') as file:
    for line in file:
        line = line.strip()
        if line:  # Add line to buffer
            buffer += line
            # Check if this is the end of a JSON object
            if line.endswith("}"):
                try:
                    data.append(json.loads(buffer))
                    buffer = ""  # Reset buffer after successful parse
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON object: {buffer}")
                    print(f"Error: {e}")
                    buffer = ""  # Reset buffer to skip the problematic object


In [4]:
data = pd.DataFrame(data)

In [5]:
data['Easting'] = data['BoundingBox'].apply(lambda x: x['Easting'])
data['Northing'] = data['BoundingBox'].apply(lambda x: x['Northing'])

In [6]:
data.drop('BoundingBox', axis=1, inplace=True)

In [7]:
data[data.duplicated(data.columns)]

,Caption,Hinweis zu den Daten,Name,Nummer,Beschreibung,Gemeindekennzeichen,Stadt/Gemeinde,Planart,Datum des Inkrafttretens,Rechtsstand (vorbehaltlich aktueller Änderungen),...,Gemeindekennzahl:,Aufstellende Gemeinde:,Planart:,Rechtsstand:,Aufstellungsbeschluss vom:,Satzungsbeschluss vom:,Datum des Inkrafttretens:,Einzelbestandteile als PDF:,Easting,Northing


In [8]:
# Convert to a GeoDataFrame
data['geometry'] = data.apply(lambda row: Point(row['Easting'], row['Northing']), axis=1)
gdf = gpd.GeoDataFrame(data, geometry='geometry', crs=3857)

In [9]:
# count duplicates of gdf
duplicates = gdf[gdf.duplicated(subset=['geometry'], keep=False)]

In [10]:
high_risk_flooding = gpd.read_file('../data/raw/flood_data/highrisk.geojson')
mid_risk_flooding = gpd.read_file('../data/raw/flood_data/midrisk.geojson')
low_risk_flooding = gpd.read_file('../data/raw/flood_data/lowrisk.geojson')

bavaria_regions = gpd.read_file('../data/raw/bavaria_regions/bav18.geojson')

In [11]:
flooding_risk_df = pd.concat([high_risk_flooding, mid_risk_flooding, low_risk_flooding])

In [12]:
data_with_flooding = gdf.sjoin(high_risk_flooding, how='left')
data_with_flooding.drop(columns=['index_right'], inplace=True)
data_with_flooding_and_bav = data_with_flooding.sjoin(bavaria_regions, how='left')

In [13]:
data_with_flooding_and_bav['flooding_risk'] = data_with_flooding_and_bav['IMPORTDATE'].apply(lambda x: 'has_flood_risk' if not pd.isnull(x) else 'no_flood_risk')

In [14]:
data_with_flooding_and_bav['year_of_bp'] = pd.to_datetime(data_with_flooding_and_bav['Datum des Inkrafttretens'], format='%d.%m.%Y').dt.year

In [15]:
data_with_flooding_and_bav = data_with_flooding_and_bav[data_with_flooding_and_bav['year_of_bp'] <= 2025]

In [16]:
median_year = data_with_flooding_and_bav['year_of_bp'].median()
print('Median of year of building permit:', median_year)

Median of year of building permit: 1998.0


In [17]:
data_with_flooding_and_bav['bplan_date_category'] = data_with_flooding_and_bav['year_of_bp'].apply(lambda x: 'older_than_median' if x < median_year else 'newer_than_median')

In [18]:
# create an id column

data_with_flooding_and_bav['id'] = data_with_flooding_and_bav.index

In [19]:
data_with_flooding_and_bav.to_file('../data/proc/building_plans/building_plans_metadata.geojson', 
                                   index=False, 
                                   driver='GeoJSON')